In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor

housing_data=pd.read_csv(r"C:/Users/000110888/OneDrive - CSULB/Desktop/housing_data.csv")

# encoding ocean_proximity
housing_data["ocean_proximity"]=housing_data["ocean_proximity"].map({
"<1H OCEAN": 1, "INLAND": 2, "NEAR BAY": 3, "NEAR OCEAN": 4, "ISLAND": 5})

# scaling target variable
housing_data["median_house_value"]=housing_data["median_house_value"]/100000

# ============================================================
# CREATING 80%/20% TRAINING/TESTING SETS
# ============================================================
train_raw, test_raw=train_test_split(housing_data, test_size=0.2, random_state=753388)

# ============================================================
# MIN-MAX SCALING OF ALL VARIABLES USING TRAINING SET ONLY
# ============================================================
train_mins=train_raw.min()
train_maxs=train_raw.max()

def min_max_scale(df, mins, maxs):
    scaled_df = df.copy()
    for col in df.columns:
        mn = mins[col]
        mx = maxs[col]
        if mx == mn:
            scaled_df[col] = 0.0
        else:
            scaled_df[col] = (df[col] - mn) / (mx - mn)
    return scaled_df


train=min_max_scale(train_raw, train_mins, train_maxs)
test=min_max_scale(test_raw, train_mins, train_maxs)

# storing train-set target scaling values for inverse transformation
y_min=train_mins["median_house_value"]
y_max=train_maxs["median_house_value"]


def unscale_y(y_scaled, y_min, y_max):
    return y_scaled*(y_max-y_min)+y_min


train_x=train.drop(columns=["median_house_value"])
train_y=train["median_house_value"]
test_x=test.drop(columns=["median_house_value"])
test_y=test["median_house_value"]


# defining function to compute accuracy within threshold
def accuracy_within(actual, predicted, pct):
    actual = np.asarray(actual)
    predicted = np.asarray(predicted)
    return np.mean(np.abs(actual-predicted)<pct*actual)

# ============================================================
# CREATING OUT-OF-FOLD SETS
# ============================================================
kf = KFold(n_splits=5, shuffle=True, random_state=559702)

oof_rf = np.full(len(train), np.nan)
oof_xgb = np.full(len(train), np.nan)
oof_svr_linear = np.full(len(train), np.nan)
oof_svr_radial = np.full(len(train), np.nan)
oof_knn = np.full(len(train), np.nan)
oof_ann = np.full(len(train), np.nan)

for train_idx, valid_idx in kf.split(train_x):
    fold_train = train.iloc[train_idx].copy()
    fold_valid = train.iloc[valid_idx].copy()

    fold_train_x = fold_train.drop(columns=["median_house_value"])
    fold_train_y = fold_train["median_house_value"]
    fold_valid_x = fold_valid.drop(columns=["median_house_value"])

    # training random forest regression
    model_rf = RandomForestRegressor(
        n_estimators=60,
        max_features=5,
        max_leaf_nodes=100,
        random_state=559702
    )
    model_rf.fit(fold_train_x, fold_train_y)
    oof_rf[valid_idx] = model_rf.predict(fold_valid_x)

    # training gradient boosting regression
    model_xgb = XGBRegressor(
        max_depth=6,
        learning_rate=0.01,
        n_estimators=1000,
        objective="reg:squarederror",
        random_state=559702,
        verbosity=0
    )
    model_xgb.fit(fold_train_x, fold_train_y)
    oof_xgb[valid_idx] = model_xgb.predict(fold_valid_x)

    # training support vector regression with linear kernel
    model_svr_linear = SVR(kernel="linear")
    model_svr_linear.fit(fold_train_x, fold_train_y)
    oof_svr_linear[valid_idx] = model_svr_linear.predict(fold_valid_x)

    # training support vector regression with radial kernel
    model_svr_radial = SVR(kernel="rbf")
    model_svr_radial.fit(fold_train_x, fold_train_y)
    oof_svr_radial[valid_idx] = model_svr_radial.predict(fold_valid_x)

    # training k-nearest neighbor regression
    knn_grid = GridSearchCV(estimator=KNeighborsRegressor(), 
        param_grid={"n_neighbors": list(range(5, 16))},
        cv=5,
        scoring="neg_mean_squared_error"
    )
    knn_grid.fit(fold_train_x, fold_train_y)
    model_knn = knn_grid.best_estimator_
    oof_knn[valid_idx] = model_knn.predict(fold_valid_x)

    # training artificial neural network
    model_ann = MLPRegressor(
        hidden_layer_sizes=(3,),
        activation="logistic",
        max_iter=2000,
        random_state=559702
    )

    try:
        model_ann.fit(fold_train_x, fold_train_y)
        ann_pred = model_ann.predict(fold_valid_x)
    except Exception:
        ann_pred = np.repeat(fold_train["median_house_value"].mean(), len(fold_valid_x))

    oof_ann[valid_idx] = ann_pred

# ============================================================
# TRAINING META-MODEL
# ============================================================
stack_train = pd.DataFrame({
    "rf": oof_rf,
    "xgb": oof_xgb,
    "svr_linear": oof_svr_linear,
    "svr_radial": oof_svr_radial,
    "knn": oof_knn,
    "ann": oof_ann,
    "median_house_value": train_y.values
})

meta_model = LinearRegression()
meta_model.fit(
    stack_train.drop(columns=["median_house_value"]),
    stack_train["median_house_value"]
)

# ============================================================
# TRAINING MODELS ON FULL TRAINING SET
# ============================================================
rf_reg = RandomForestRegressor(
    n_estimators=60,
    max_features=5,
    max_leaf_nodes=100,
    random_state=753388
)
rf_reg.fit(train_x, train_y)

xgb_reg = XGBRegressor(
    max_depth=6,
    learning_rate=0.01,
    n_estimators=1000,
    objective="reg:squarederror",
    random_state=753388,
    verbosity=0
)
xgb_reg.fit(train_x, train_y)

svr_linear = SVR(kernel="linear")
svr_linear.fit(train_x, train_y)

svr_radial = SVR(kernel="rbf")
svr_radial.fit(train_x, train_y)

knn_grid_full = GridSearchCV(
    estimator=KNeighborsRegressor(),
    param_grid={"n_neighbors": list(range(5, 16))},
    cv=5,
    scoring="neg_mean_squared_error"
)
knn_grid_full.fit(train_x, train_y)
knn_reg = knn_grid_full.best_estimator_

ann_reg = MLPRegressor(
    hidden_layer_sizes=(3,),
    activation="logistic",
    max_iter=2000,
    random_state=753388
)
ann_reg.fit(train_x, train_y)

# ============================================================
# PREDICTING ON TEST SET
# ============================================================
test_rf=rf_reg.predict(test_x)
test_xgb=xgb_reg.predict(test_x)
test_svr_linear=svr_linear.predict(test_x)
test_svr_radial=svr_radial.predict(test_x)
test_knn=knn_reg.predict(test_x)
test_ann=ann_reg.predict(test_x)

stack_test = pd.DataFrame({
    "rf": test_rf,
    "xgb": test_xgb,
    "svr_linear": test_svr_linear,
    "svr_radial": test_svr_radial,
    "knn": test_knn,
    "ann": test_ann
})

# computing stacked predictions
stack_pred_scaled=meta_model.predict(stack_test)

actual_y=unscale_y(test_y, y_min, y_max)
pred_y=unscale_y(stack_pred_scaled, y_min, y_max)

# ============================================================
# COMPUTING ACCURACY WITHIN 10%, 15%, AND 20%
# ============================================================
acc10=accuracy_within(actual_y, pred_y, 0.10)
acc15=accuracy_within(actual_y, pred_y, 0.15)
acc20=accuracy_within(actual_y, pred_y, 0.20)

# ============================================================
# COMBINING OUTPUTS INTO ONE TABLE
# ============================================================
# unscaling individual model predictions
pred_rf=unscale_y(test_rf, y_min, y_max)
pred_xgb=unscale_y(test_xgb, y_min, y_max)
pred_svr_linear=unscale_y(test_svr_linear, y_min, y_max)
pred_svr_radial=unscale_y(test_svr_radial, y_min, y_max)
pred_knn=unscale_y(test_knn, y_min, y_max)
pred_ann=unscale_y(test_ann, y_min, y_max)

# unscaling stacked predictions
stack_pred=unscale_y(stack_pred_scaled, y_min, y_max)

# combining outputs into one table
results = pd.DataFrame({
    "Model": [
        "Random Forest",
        "XGBoost",
        "SVR Linear",
        "SVR Radial",
        "KNN",
        "ANN",
        "Stacked"
    ],
    "Acc_10": [
        accuracy_within(actual_y, pred_rf, 0.10),
        accuracy_within(actual_y, pred_xgb, 0.10),
        accuracy_within(actual_y, pred_svr_linear, 0.10),
        accuracy_within(actual_y, pred_svr_radial, 0.10),
        accuracy_within(actual_y, pred_knn, 0.10),
        accuracy_within(actual_y, pred_ann, 0.10),
        accuracy_within(actual_y, stack_pred, 0.10)
    ],
    "Acc_15": [
        accuracy_within(actual_y, pred_rf, 0.15),
        accuracy_within(actual_y, pred_xgb, 0.15),
        accuracy_within(actual_y, pred_svr_linear, 0.15),
        accuracy_within(actual_y, pred_svr_radial, 0.15),
        accuracy_within(actual_y, pred_knn, 0.15),
        accuracy_within(actual_y, pred_ann, 0.15),
        accuracy_within(actual_y, stack_pred, 0.15)
    ],
    "Acc_20": [
        accuracy_within(actual_y, pred_rf, 0.20),
        accuracy_within(actual_y, pred_xgb, 0.20),
        accuracy_within(actual_y, pred_svr_linear, 0.20),
        accuracy_within(actual_y, pred_svr_radial, 0.20),
        accuracy_within(actual_y, pred_knn, 0.20),
        accuracy_within(actual_y, pred_ann, 0.20),
        accuracy_within(actual_y, stack_pred, 0.20)
    ]
})

results["Acc_10"] = results["Acc_10"].round(4)
results["Acc_15"] = results["Acc_15"].round(4)
results["Acc_20"] = results["Acc_20"].round(4)

print(results)